# Z-filter example (ZTF and ZSOS)

This notebook checks that pyFDN’s **ZTF** (transfer function) and **ZSOS** (second-order sections) filter classes match `scipy.signal.freqz` / `sosfreqz`, and that their frequency-derivative (`.der`) matches a numerical derivative.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import freqz, sosfreqz

import pyFDN


def plot_lines(w, h, hh, dh, dhh, title_suffix="", offset = 0.0):
    """Plot frequency response and derivative: scipy vs zFilter."""
    plt.figure()
    plt.plot(w, np.real(h), label="Real (freqz)", linewidth=4)
    plt.plot(w, np.imag(h), label="Imag (freqz)", linewidth=4)
    plt.plot(w, np.real(hh) + offset, label="Real (zFilter)", linestyle="--", linewidth=1.5)
    plt.plot(w, np.imag(hh) + offset, label="Imag (zFilter)", linestyle="--", linewidth=1.5)
    plt.grid(True)
    plt.legend()
    plt.xlabel("Frequency [rad]")
    plt.ylabel("Amplitude [lin]")
    plt.title(f"Frequency response {title_suffix}")

    plt.figure()
    plt.plot(w[1:], np.real(dh), label="Real (freqz deriv)", linewidth=4)
    plt.plot(w[1:], np.imag(dh), label="Imag (freqz deriv)", linewidth=4)
    plt.plot(w, np.real(dhh) + offset, label="Real (zFilter deriv)", linestyle="--", linewidth=1.5)
    plt.plot(w, np.imag(dhh) + offset, label="Imag (zFilter deriv)", linestyle="--", linewidth=1.5)
    plt.grid(True)
    plt.legend()
    plt.xlabel("Frequency [rad]")
    plt.ylabel("Amplitude [lin]")
    plt.title(f"Derivative {title_suffix}")


def probe_z_filter(z_filter, ww):
    """Evaluate filter and its derivative at frequencies ww (unit circle)."""
    hh = np.zeros_like(ww, dtype=np.complex128)
    dhh = np.zeros_like(ww, dtype=np.complex128)
    for i, w_val in enumerate(ww):
        mat_at = z_filter.at(w_val)
        hh[i] = mat_at[0, 0] if mat_at.ndim == 2 else mat_at[0]
        mat_der = z_filter.der(w_val)
        dhh[i] = mat_der[0, 0] if mat_der.ndim == 2 else mat_der[0]
    return hh, dhh

## ZTF (transfer function)

Compare `pyFDN.ZTF` frequency response and derivative to `scipy.signal.freqz` and a numerical derivative.

In [ ]:
np.random.seed(2)
n_fft = 2**14

n, m, order = 2, 1, 5
a = np.random.randn(n, m, order)
b = np.random.randn(n, m, order)

b_1d = b[0, 0, :].squeeze()
a_1d = a[0, 0, :].squeeze()
w, h = freqz(b_1d, a_1d, worN=n_fft)
ww = np.exp(1j * w)
dh = np.diff(h) / np.diff(ww)

z_filter = pyFDN.ZTF(b, a, is_diagonal=True)
hh, dhh = probe_z_filter(z_filter, ww)

plot_lines(w, h, hh, dh, dhh, "(ZTF)")

print(f"ZTF frequency response max error: {np.max(np.abs(h - hh))}")
print(f"ZTF derivative max error: {np.max(np.abs(dh - dhh[1:]))}")
assert pyFDN.is_almost_zero(h - hh), "ZTF frequency response mismatch"
assert pyFDN.is_almost_zero(dh - dhh[1:], tol=0.15), "ZTF derivative mismatch"
print("ZTF tests passed.")

## ZSOS (second-order sections)

Same comparison for `pyFDN.ZSOS` vs `scipy.signal.sosfreqz`.

In [ ]:
plt.close("all")
n_sos = 3
n, m = 2, 1
sos = np.random.rand(n, m, n_sos, 6)

# Normalize SOS so leading denominator coefficient is 1 (for scipy)
for i in range(n_sos):
    a0 = sos[0, 0, i, 3]
    if a0 != 0:
        sos[0, 0, i, :3] /= a0
        sos[0, 0, i, 3:] /= a0

sos_1d = sos[0, 0, :, :].squeeze()
w, h = sosfreqz(sos_1d, worN=n_fft)
ww = np.exp(1j * w)
dh = np.diff(h) / np.diff(ww)

z_filter = pyFDN.ZSOS(sos, is_diagonal=False)
hh, dhh = probe_z_filter(z_filter, ww)

plot_lines(w, h, hh, dh, dhh, "(ZSOS)")

print(f"ZSOS frequency response max error: {np.max(np.abs(h - hh))}")
print(f"ZSOS derivative max error: {np.max(np.abs(dh - dhh[1:]))}")
assert pyFDN.is_almost_zero(h - hh), "ZSOS frequency response mismatch"
assert pyFDN.is_almost_zero(dh - dhh[1:], tol=0.15), "ZSOS derivative mismatch"
print("ZSOS tests passed.")

plt.show()